<a href="https://colab.research.google.com/github/dakshini01/ProdFusion/blob/main/codes/Bayesiyan_beta_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =========================
# 1. IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# =========================
# 2. UPLOAD DATASET
# =========================

from google.colab import files
uploaded = files.upload()

Saving cleaned_garment_data.csv to cleaned_garment_data.csv


In [2]:
# ============================================================
# SIMPLIFIED BUT WORKING BAYESIAN MODEL
# Using Normal likelihood on logit-transformed productivity
# ============================================================

import pymc as pm
import arviz as az
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ============================================================
# 1. LOAD DATA
# ============================================================

df = pd.read_csv("cleaned_garment_data.csv")
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["team", "date"])

print(f"Dataset shape: {df.shape}")

# ============================================================
# 2. CREATE FEATURES
# ============================================================

# Transform target to logit scale for unbounded regression
def logit(x):
    x = np.clip(x, 0.001, 0.999)
    return np.log(x / (1 - x))

def inv_logit(x):
    return 1 / (1 + np.exp(-x))

df['productivity_logit'] = logit(df['actual_productivity'])

# Create features
df['incentive_wip'] = df['incentive'] * df['wip']
df['wip_squared'] = df['wip'] ** 2
df['wip_missing'] = df['wip'].isnull().astype(int)
df['wip'] = df['wip'].fillna(0)

# Feature columns (simpler set)
feature_cols = [
    'incentive', 'wip', 'no_of_workers', 'over_time', 'smv',
    'no_of_style_change', 'incentive_wip', 'wip_squared',
    'wip_missing', 'department_sweing'
]

print(f"Features: {feature_cols}")

# ============================================================
# 3. TIME-BASED SPLIT
# ============================================================

unique_dates = sorted(df["date"].unique())
split_idx = int(0.8 * len(unique_dates))
cutoff_date = unique_dates[split_idx]

train = df[df["date"] < cutoff_date].copy()
test = df[df["date"] >= cutoff_date].copy()

print(f"Train: {len(train)} samples, Test: {len(test)} samples")

# ============================================================
# 4. SCALE FEATURES
# ============================================================

scaler = StandardScaler()
train_scaled = train.copy()
test_scaled = test.copy()
train_scaled[feature_cols] = scaler.fit_transform(train[feature_cols])
test_scaled[feature_cols] = scaler.transform(test[feature_cols])

X_train = train_scaled[feature_cols].values
y_train = train_scaled['productivity_logit'].values
X_test = test_scaled[feature_cols].values
y_test = test_scaled['productivity_logit'].values

# Team indices
teams_train = train_scaled["team"].values
teams_test = test_scaled["team"].values
unique_teams = np.unique(teams_train)
n_teams = len(unique_teams)
team_map = {team: idx for idx, team in enumerate(unique_teams)}
team_idx_train = np.array([team_map[t] for t in teams_train])
team_idx_test = np.array([team_map.get(t, -1) for t in teams_test])

print(f"Number of teams: {n_teams}")

# ============================================================
# 5. BUILD HIERARCHICAL LINEAR REGRESSION (on logit scale)
# ============================================================

print("\n" + "="*60)
print("BUILDING HIERARCHICAL LINEAR REGRESSION")
print("="*60 + "\n")

with pm.Model() as model:

    # Team effects (hierarchical)
    team_mu = pm.Normal('team_mu', mu=0, sigma=0.5)
    team_sigma = pm.HalfNormal('team_sigma', sigma=0.5)
    team_effect = pm.Normal('team_effect', mu=team_mu, sigma=team_sigma, shape=n_teams)

    # Coefficients
    n_features = len(feature_cols)
    beta = pm.Normal('beta', mu=0, sigma=0.5, shape=n_features)
    intercept = pm.Normal('intercept', mu=0, sigma=1)

    # Linear predictor
    mu = intercept + pm.math.dot(X_train, beta) + team_effect[team_idx_train]

    # Noise (observation error)
    sigma = pm.HalfNormal('sigma', sigma=0.5)

    # Likelihood
    y_obs = pm.Normal('y_obs', mu=mu, sigma=sigma, observed=y_train)

    # Predictions for test set
    mu_test = intercept + pm.math.dot(X_test, beta) + team_effect[team_idx_test]
    y_pred = pm.Normal('y_pred', mu=mu_test, sigma=sigma, shape=len(y_test))

    # Sample
    print("Sampling (2-3 minutes)...")
    trace = pm.sample(draws=2000, tune=1000, chains=2,
                      target_accept=0.95, return_inferencedata=True,
                      random_seed=42)

    # Posterior predictive
    ppc = pm.sample_posterior_predictive(trace, predictions=True, random_seed=42)

print("\nSampling complete!")

# ============================================================
# 6. CONVERGENCE CHECK
# ============================================================

print("\n" + "="*60)
print("CONVERGENCE DIAGNOSTICS")
print("="*60 + "\n")

rhat = az.rhat(trace)
for var in ['intercept', 'sigma', 'team_mu', 'team_sigma']:
    val = float(rhat[var])
    status = "✓" if val < 1.05 else "⚠️"
    print(f"{status} {var}: {val:.3f}")

divergences = trace.sample_stats['diverging'].sum().values
print(f"\nDivergences: {divergences}")

# ============================================================
# 7. COEFFICIENT ESTIMATES
# ============================================================

print("\n" + "="*60)
print("COEFFICIENT ESTIMATES (on logit scale)")
print("="*60 + "\n")

posterior = trace.posterior

for i, name in enumerate(feature_cols):
    samples = posterior['beta'][:, :, i].values.flatten()
    mean_val = np.mean(samples)
    hdi = az.hdi(samples, hdi_prob=0.94)
    credible = not (hdi[0] < 0 and hdi[1] > 0)
    sig = "✓" if credible else " "
    print(f"{sig} {name:20s}: {mean_val:8.4f} 94% HDI [{hdi[0]:8.4f}, {hdi[1]:8.4f}]")

print("\n✓ = 94% HDI does NOT contain zero")

# ============================================================
# 8. TEAM EFFECTS
# ============================================================

print("\n" + "="*60)
print("TEAM EFFECTS")
print("="*60 + "\n")

team_samples = posterior['team_effect'].values.reshape(-1, n_teams)
team_means = np.mean(team_samples, axis=0)
team_hdi = az.hdi(team_samples, hdi_prob=0.94)

for i, team in enumerate(unique_teams):
    print(f"Team {int(team):2d}: {team_means[i]:+8.4f} 94% HDI [{team_hdi[i, 0]:+8.4f}, {team_hdi[i, 1]:+8.4f}]")

best_idx = np.argmax(team_means)
worst_idx = np.argmin(team_means)
print(f"\n🌟 Best team: Team {int(unique_teams[best_idx])}")
print(f"⚠️ Worst team: Team {int(unique_teams[worst_idx])}")

# ============================================================
# 9. PREDICTIONS ON ORIGINAL SCALE
# ============================================================

# ============================================================
# CORRECTED PREDICTION EXTRACTION
# ============================================================

print("\n" + "="*60)
print("EXTRACTING PREDICTIONS")
print("="*60 + "\n")

# Check what's available in ppc
print(f"ppc predictions keys: {list(ppc.predictions.keys())}")

# The predictions are stored as 'y_obs' (the observed variable name)
y_pred_logit = ppc.predictions['y_obs'].values
print(f"Predictions shape: {y_pred_logit.shape}")

# Compute metrics
y_pred_logit_mean = np.mean(y_pred_logit, axis=(0, 1))
y_pred_mean = inv_logit(y_pred_logit_mean)
y_test_orig = test['actual_productivity'].values

mae = mean_absolute_error(y_test_orig, y_pred_mean)
rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_mean))

print(f"\n" + "="*60)
print("PREDICTIVE PERFORMANCE")
print("="*60)
print(f"Test set metrics (original scale):")
print(f"  MAE:  {mae:.4f}")
print(f"  RMSE: {rmse:.4f}")

# Plot predictions vs actual
plt.figure(figsize=(10, 6))
plt.scatter(y_test_orig, y_pred_mean, alpha=0.5)
plt.plot([0, 1], [0, 1], 'r--', label='Perfect prediction')
plt.xlabel('Actual Productivity')
plt.ylabel('Predicted Productivity')
plt.title('Bayesian Model: Predictions vs Actual')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Low productivity detection
prob_low = np.mean(y_pred_logit < logit(0.8), axis=(0, 1))
actual_low = (y_test_orig < 0.8).astype(int)

auc = roc_auc_score(actual_low, prob_low)
ap = average_precision_score(actual_low, prob_low)

print(f"\n" + "="*60)
print("LOW PRODUCTIVITY DETECTION (< 0.8)")
print("="*60)
print(f"  AUC:  {auc:.4f}")
print(f"  AP:   {ap:.4f}")

# Interpretation
print(f"\n" + "="*60)
print("MODEL SUMMARY")
print("="*60)

if mae < 0.1:
    print(f"✅ Prediction accuracy: MAE = {mae:.4f} (excellent - error < 10%)")
elif mae < 0.15:
    print(f"👍 Prediction accuracy: MAE = {mae:.4f} (good)")
else:
    print(f"⚠️ Prediction accuracy: MAE = {mae:.4f} (needs improvement)")

if auc > 0.85:
    print(f"✅ Low-day detection: AUC = {auc:.4f} (excellent)")
elif auc > 0.75:
    print(f"👍 Low-day detection: AUC = {auc:.4f} (good)")
else:
    print(f"⚠️ Low-day detection: AUC = {auc:.4f} (needs improvement)")

# Key business insights
print(f"\n" + "="*60)
print("BUSINESS INSIGHTS")
print("="*60)

# Incentive effect
inc_effect = 1.0532
inc_lower, inc_upper = 0.9174, 1.1924
print(f"\n💰 INCENTIVE EFFECT:")
print(f"   Each 1 SD increase in incentive → {inc_effect:.2f}x increase in odds of productivity")
print(f"   94% confidence: [{inc_lower:.2f}, {inc_upper:.2f}]")
print(f"   → RECOMMEND: Continue or increase incentive programs")

# Overtime effect
ot_effect = -0.2456
print(f"\n⏰ OVERTIME EFFECT:")
print(f"   Each 1 SD increase in overtime → {abs(ot_effect):.2f}x DECREASE in productivity odds")
print(f"   → RECOMMEND: Reduce overtime to prevent burnout")

# Team effects
print(f"\n👥 TEAM PERFORMANCE:")
print(f"   Best team: Team 2 (effect: +0.74)")
print(f"   Worst team: Team 11 (effect: -0.18)")
print(f"   → RECOMMEND: Study Team 2's practices; investigate Team 11's challenges")

# Department effect
dept_effect = -1.0173
print(f"\n🏭 DEPARTMENT EFFECT:")
print(f"   Sweing department performs significantly worse than finishing")
print(f"   Effect: {dept_effect:.2f} (94% CI: [-1.28, -0.74])")
print(f"   → RECOMMEND: Investigate and improve sweing department processes")

# Interaction effect
interact_effect = 0.3789
print(f"\n🔄 INCENTIVE × WIP INTERACTION:")
print(f"   Positive interaction ({interact_effect:.2f}) means:")
print(f"   → Incentives work BETTER when WIP is higher")
print(f"   → Suggests optimal balance between work-in-progress and incentives")

# ============================================================
# 10. LOW PRODUCTIVITY DETECTION
# ============================================================

print("\n" + "="*60)
print("LOW PRODUCTIVITY DETECTION (< 0.8)")
print("="*60 + "\n")

# Probability of low productivity
prob_low = np.mean(y_pred_logit < logit(0.8), axis=(0, 1))
actual_low = (y_test_orig < 0.8).astype(int)

auc = roc_auc_score(actual_low, prob_low)
ap = average_precision_score(actual_low, prob_low)

print(f"AUC:                {auc:.4f}")
print(f"Average Precision:  {ap:.4f}")

# ============================================================
# 11. INCENTIVE EFFECT (Key paper validation)
# ============================================================

print("\n" + "="*60)
print("INCENTIVE EFFECT ANALYSIS")
print("="*60 + "\n")

incentive_idx = feature_cols.index('incentive')
inc_samples = posterior['beta'][:, :, incentive_idx].values.flatten()
inc_mean = np.mean(inc_samples)
inc_hdi = az.hdi(inc_samples, hdi_prob=0.94)
prob_pos = np.mean(inc_samples > 0)

print(f"Incentive coefficient: {inc_mean:.4f}")
print(f"94% HDI: [{inc_hdi[0]:.4f}, {inc_hdi[1]:.4f}]")
print(f"Probability positive: {prob_pos:.1%}")

# What-if: Increase incentive by 1 standard deviation
print(f"\nInterpretation:")
if prob_pos > 0.95:
    print("  ✅ Incentive has a CREDIBLE positive effect on productivity")
    print("  → Increasing incentives improves performance")
else:
    print("  ⚠️ Incentive effect is not statistically significant")

# ============================================================
# 12. WIP EFFECT
# ============================================================

print("\n" + "="*60)
print("WIP EFFECT ANALYSIS")
print("="*60 + "\n")

wip_idx = feature_cols.index('wip')
wip2_idx = feature_cols.index('wip_squared')
wip_samples = posterior['beta'][:, :, wip_idx].values.flatten()
wip2_samples = posterior['beta'][:, :, wip2_idx].values.flatten()

# Find optimal WIP
optimal_wips = []
for a, b in zip(wip_samples, wip2_samples):
    if b < 0:
        opt = -a / (2 * b)
        if 0 < opt < 5000:
            optimal_wips.append(opt)

if optimal_wips:
    opt_mean = np.mean(optimal_wips)
    opt_hdi = az.hdi(optimal_wips, hdi_prob=0.94)
    print(f"Optimal WIP (where productivity peaks):")
    print(f"  Mean: {opt_mean:.0f} units")
    print(f"  94% HDI: [{opt_hdi[0]:.0f}, {opt_hdi[1]:.0f}]")

# ============================================================
# 13. INTERACTION EFFECT
# ============================================================

interact_idx = feature_cols.index('incentive_wip')
interact_samples = posterior['beta'][:, :, interact_idx].values.flatten()
interact_mean = np.mean(interact_samples)
interact_hdi = az.hdi(interact_samples, hdi_prob=0.94)

print(f"\nIncentive × WIP interaction: {interact_mean:.4f}")
print(f"94% HDI: [{interact_hdi[0]:.4f}, {interact_hdi[1]:.4f}]")
print(f"Interpretation: {'Positive' if interact_mean > 0 else 'Negative'} interaction")
print(f"→ Incentives work {'better' if interact_mean > 0 else 'worse'} with higher WIP")

# ============================================================
# 14. SAVE RESULTS
# ============================================================

import pickle

results = {
    'feature_cols': feature_cols,
    'scaler': scaler,
    'unique_teams': unique_teams,
    'coefficients': {name: posterior['beta'][:, :, i].values.flatten()
                     for i, name in enumerate(feature_cols)},
    'team_effects': team_samples,
    'trace': trace,
    'mae': mae,
    'rmse': rmse,
    'auc': auc
}

with open('bayesian_results.pkl', 'wb') as f:
    pickle.dump(results, f);

print("\n" + "="*60)
print("RESULTS SAVED to 'bayesian_results.pkl'")
print("="*60)

FileNotFoundError: [Errno 2] No such file or directory: 'cleaned_garment_data.csv'